# Sentiment Analysis on SST-2 with OpenAI API

This notebook loads the SST-2 dataset from the GLUE benchmark, samples a small evaluation subset, uses an OpenAI model for zero-shot sentiment classification, and evaluates performance.

## 1. Setup

Install dependencies if needed, then import libraries and configure the OpenAI client.

In [ ]:
# If running in a fresh environment, uncomment the next line:
# !pip install -q datasets pandas scikit-learn openai matplotlib seaborn

In [ ]:
import os
import re
import json
import time
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

from openai import OpenAI

In [ ]:
pd.set_option('display.max_colwidth', 200)
sns.set_style('whitegrid')

In [ ]:
# Configure your API key before running:
# os.environ['OPENAI_API_KEY'] = 'your-api-key'

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('OPENAI_API_KEY environment variable is not set.')

client = OpenAI(api_key=api_key)

# Choose a model available to your account.
MODEL_NAME = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
MODEL_NAME

## 2. Load and inspect the SST-2 dataset

In [ ]:
dataset = load_dataset('glue', 'sst2')
dataset

In [ ]:
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()

label_map = {0: 'negative', 1: 'positive'}
train_df['label_name'] = train_df['label'].map(label_map)
val_df['label_name'] = val_df['label'].map(label_map)

print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('\nTrain sample:')
display(train_df.head())
print('\nValidation sample:')
display(val_df.head())

In [ ]:
print('Train label distribution:')
display(train_df['label_name'].value_counts().rename_axis('label').reset_index(name='count'))

print('Validation label distribution:')
display(val_df['label_name'].value_counts().rename_axis('label').reset_index(name='count'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=train_df, x='label_name', order=['negative', 'positive'], ax=axes[0])
axes[0].set_title('Train Label Distribution')
axes[0].set_xlabel('Label')

sns.countplot(data=val_df, x='label_name', order=['negative', 'positive'], ax=axes[1])
axes[1].set_title('Validation Label Distribution')
axes[1].set_xlabel('Label')

plt.tight_layout()
plt.show()

## 3. Prepare a small evaluation subset

For a lightweight workflow, sample a balanced subset from the validation set.

In [ ]:
SAMPLES_PER_CLASS = 25
RANDOM_STATE = 42

eval_df = (
    val_df.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(n=min(SAMPLES_PER_CLASS, len(x)), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

eval_df['label_name'] = eval_df['label'].map(label_map)
eval_df = eval_df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print('Evaluation subset shape:', eval_df.shape)
display(eval_df.head(10))

In [ ]:
eval_df['label_name'].value_counts()

## 4. Prompt design and helper functions

We use a zero-shot prompt and require the model to return only one label: `positive` or `negative`.

In [ ]:
SYSTEM_PROMPT = (
    'You are a sentiment classifier for movie review sentences. '
    'Return only one word: positive or negative.'
)

def build_prompt(sentence: str) -> str:
    return (
        'Classify the sentiment of the following movie review sentence. '\
        'Respond with only one word: positive or negative.\n\n'
        f'Sentence: {sentence}'
    )

def parse_prediction(text: str):
    if text is None:
        return None
    cleaned = text.strip().lower()
    cleaned = re.sub(r'[^a-z]+', ' ', cleaned).strip()
    if 'positive' in cleaned:
        return 'positive'
    if 'negative' in cleaned:
        return 'negative'
    return None

def label_name_to_id(label_name: str):
    inverse = {'negative': 0, 'positive': 1}
    return inverse.get(label_name)

def classify_sentence(sentence: str, model: str = MODEL_NAME, max_retries: int = 3, sleep_seconds: float = 1.0):
    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=model,
                input=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': build_prompt(sentence)}
                ],
                temperature=0
            )
            raw_text = response.output_text.strip() if hasattr(response, 'output_text') else ''
            pred = parse_prediction(raw_text)
            return raw_text, pred, None
        except Exception as e:
            last_error = str(e)
            time.sleep(sleep_seconds * (attempt + 1))
    return None, None, last_error

In [ ]:
test_sentences = [
    'A delightful and moving film with wonderful performances.',
    'A dull, predictable mess that never becomes interesting.',
    'It tries hard but ends up feeling flat and overly long.'
]

test_results = []
for s in test_sentences:
    raw, pred, err = classify_sentence(s)
    test_results.append({'sentence': s, 'raw_response': raw, 'parsed_prediction': pred, 'error': err})

display(pd.DataFrame(test_results))

## 5. Run inference with the OpenAI API

In [ ]:
results = []

for idx, row in eval_df.iterrows():
    sentence = row['sentence']
    true_label = row['label']
    true_label_name = row['label_name']
    raw_response, pred_label_name, error = classify_sentence(sentence)
    pred_label = label_name_to_id(pred_label_name) if pred_label_name is not None else None

    results.append({
        'idx': idx,
        'sentence': sentence,
        'true_label': true_label,
        'true_label_name': true_label_name,
        'raw_response': raw_response,
        'pred_label_name': pred_label_name,
        'pred_label': pred_label,
        'error': error
    })

    if (idx + 1) % 10 == 0 or (idx + 1) == len(eval_df):
        print(f'Processed {idx + 1}/{len(eval_df)}')

results_df = pd.DataFrame(results)
display(results_df.head())

In [ ]:
print('Prediction distribution:')
display(results_df['pred_label_name'].value_counts(dropna=False).rename_axis('prediction').reset_index(name='count'))

print('Errors / failed calls:')
display(results_df[results_df['error'].notna()].head())

print('Ambiguous / unparsable outputs:')
display(results_df[results_df['pred_label_name'].isna()].head())

## 6. Evaluate model performance

In [ ]:
valid_results_df = results_df.dropna(subset=['pred_label']).copy()

if len(valid_results_df) == 0:
    raise ValueError('No valid predictions were parsed. Check API responses or parsing logic.')

y_true = valid_results_df['true_label']
y_pred = valid_results_df['pred_label']

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

metrics_df = pd.DataFrame([
    {'metric': 'accuracy', 'value': accuracy},
    {'metric': 'precision', 'value': precision},
    {'metric': 'recall', 'value': recall},
    {'metric': 'f1', 'value': f1},
    {'metric': 'coverage', 'value': len(valid_results_df) / len(results_df)}
])

display(metrics_df)

print('Classification report:')
print(classification_report(y_true, y_pred, target_names=['negative', 'positive']))

In [ ]:
cm_df = pd.DataFrame(cm, index=['true_negative', 'true_positive'], columns=['pred_negative', 'pred_positive'])
display(cm_df)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

## 7. Analyze errors and discuss improvements

In [ ]:
analysis_df = results_df.copy()
analysis_df['correct'] = analysis_df['true_label'] == analysis_df['pred_label']

misclassified_df = analysis_df[(analysis_df['pred_label'].notna()) & (~analysis_df['correct'])].copy()
print(f'Misclassified examples: {len(misclassified_df)}')
display(misclassified_df[['sentence', 'true_label_name', 'pred_label_name', 'raw_response']].head(20))

In [ ]:
summary = {
    'model': MODEL_NAME,
    'n_examples_total': int(len(results_df)),
    'n_valid_predictions': int(len(valid_results_df)),
    'n_failed_or_unparsed': int(len(results_df) - len(valid_results_df)),
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1)
}

print(json.dumps(summary, indent=2))

### Suggested next steps

- Refine the prompt with a few-shot format using labeled SST-2-like examples.
- Increase the evaluation subset or run on the full validation set for more stable estimates.
- Add robust response validation and retry logic for non-conforming outputs.
- Compare zero-shot OpenAI performance against a fine-tuned HuggingFace baseline.
- Investigate recurring failure modes such as negation, sarcasm, mixed sentiment, and ambiguous phrasing.